# CineMatch — Preprocessing Notebook
Run all cells once to generate `movies_new.pkl` and `similarity_new.pkl`.

**Weights used:** Genres ×5 · Keywords ×4 · Director ×3 · Cast ×2 · Overview ×1

In [1]:
import pandas as pd
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ── Load dataset ─────────────────────────────────────────────────────────────
df = pd.read_csv('tmdb_movies_data.csv')
print(f'Loaded {len(df):,} rows')
print('Columns:', df.columns.tolist())

Loaded 10,866 rows
Columns: ['id', 'imdb_id', 'popularity', 'budget', 'revenue', 'original_title', 'cast', 'homepage', 'director', 'tagline', 'keywords', 'overview', 'runtime', 'genres', 'production_companies', 'release_date', 'vote_count', 'vote_average', 'release_year', 'budget_adj', 'revenue_adj']


In [2]:
# ── Select & clean columns ───────────────────────────────────────────────────
cols = [
    'id', 'original_title', 'genres', 'keywords', 'cast', 'director',
    'overview', 'runtime', 'vote_average', 'release_year',
    'budget', 'revenue', 'production_companies', 'tagline'
]
df = df[[c for c in cols if c in df.columns]].copy()
df.dropna(subset=['overview'], inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'After cleaning: {len(df):,} movies')
df.head(3)

After cleaning: 10,862 movies


,id,original_title,genres,keywords,cast,director,overview,runtime,vote_average,release_year,budget,revenue,production_companies,tagline
0,135397,Jurassic World,Action|Adventure|Science Fiction|Thriller,monster|dna|tyrannosaurus rex|velociraptor|island,Chris Pratt|Bryce Dallas Howard|Irrfan Khan|Vi...,Colin Trevorrow,Twenty-two years after the events of Jurassic ...,124,6.5,2015,150000000,1513528810,Universal Studios|Amblin Entertainment|Legenda...,The park is open.
1,76341,Mad Max: Fury Road,Action|Adventure|Science Fiction|Thriller,future|chase|post-apocalyptic|dystopia|australia,Tom Hardy|Charlize Theron|Hugh Keays-Byrne|Nic...,George Miller,An apocalyptic story set in the furthest reach...,120,7.1,2015,150000000,378436354,Village Roadshow Pictures|Kennedy Miller Produ...,What a Lovely Day.
2,262500,Insurgent,Adventure|Science Fiction|Thriller,based on novel|revolution|dystopia|sequel|dyst...,Shailene Woodley|Theo James|Kate Winslet|Ansel...,Robert Schwentke,Beatrice Prior must confront her inner demons ...,119,6.3,2015,110000000,295238201,Summit Entertainment|Mandeville Films|Red Wago...,One Choice Can Destroy You


In [3]:
# ── Build weighted tag string ────────────────────────────────────────────────
# Genres × 5  |  Keywords × 4  |  Director × 3  |  Cast × 2  |  Overview × 1
def make_tags(row):
    g = str(row.get('genres', '')).replace('|', ' ')
    k = str(row.get('keywords', '')).replace('|', ' ')
    d = str(row.get('director', '')).replace(' ', '_')
    c = ' '.join(str(row.get('cast', '')).replace('|', ' ').split()[:5])
    o = str(row.get('overview', ''))
    return (g + ' ') * 5 + (k + ' ') * 4 + (d + ' ') * 3 + (c + ' ') * 2 + o

df['tags'] = df.apply(make_tags, axis=1)
print('Sample tag for Fight Club:')
fc = df[df['original_title'] == 'Fight Club']
if len(fc): print(fc['tags'].values[0][:200])

Sample tag for Fight Club:
Drama Drama Drama Drama Drama support group dual identity nihilism rage and hate insomnia support group dual identity nihilism rage and hate insomnia support group dual identity nihilism rage and hate


In [4]:
# ── TF-IDF vectorisation ─────────────────────────────────────────────────────
tfidf = TfidfVectorizer(
    max_features=12000,
    stop_words='english',
    ngram_range=(1, 2),
    min_df=2          # ignore tokens that appear in only 1 movie (noise)
)
matrix = tfidf.fit_transform(df['tags'])
print(f'Matrix shape: {matrix.shape}  |  Vocab: {len(tfidf.vocabulary_):,} terms')

Matrix shape: (10862, 12000)  |  Vocab: 12,000 terms


In [5]:
# ── Cosine similarity (sparse) ───────────────────────────────────────────────
similarity = cosine_similarity(matrix, dense_output=False)
print(f'Similarity matrix: {similarity.shape}')

Similarity matrix: (10862, 10862)


In [6]:
# ── Quick sanity check ───────────────────────────────────────────────────────
def top_matches(title, n=8):
    matches = df[df['original_title'] == title]
    if matches.empty:
        print(f'{title} not found'); return
    idx = matches.index[0]
    scores = sorted(enumerate(similarity[idx].toarray().flatten()),
                    key=lambda x: x[1], reverse=True)
    print(f'\nTop {n} for "{title}":')
    for i, s in scores[1:n+1]:
        r = df.iloc[i]
        print(f"  {r['original_title']:40s}  {s:.3f}  {r['genres']}")

top_matches('The Dark Knight')
top_matches('Inception')
top_matches('Shutter Island')


Top 8 for "The Dark Knight":
  Batman Begins                             0.711  Action|Crime|Drama
  The Dark Knight Rises                     0.643  Action|Crime|Drama|Thriller
  Kick-Ass                                  0.426  Action|Crime
  Batman Returns                            0.396  Action|Crime|Fantasy|Science Fiction|Thriller
  Tiger House                               0.377  Thriller|Drama|Action|Crime
  Kick-Ass 2                                0.341  Action|Adventure|Crime
  Kidnapping Mr. Heineken                   0.339  Drama|Action|Crime|Thriller
  Batman                                    0.313  Fantasy|Action

Top 8 for "Inception":
  Inception: The Cobol Job                  0.434  Animation|Action|Thriller|Science Fiction
  Residue                                   0.345  Science Fiction|Mystery|Horror|Thriller
  Enter Nowhere                             0.340  Horror|Thriller|Science Fiction|Mystery
  American Warships                         0.337  Action|Thril

In [7]:
# ── Save pickle files ────────────────────────────────────────────────────────
pickle.dump(df,         open('movies_new.pkl',      'wb'))
pickle.dump(similarity, open('similarity_new.pkl',  'wb'))
print('movies_new.pkl      saved:', df.shape)
print('similarity_new.pkl  saved:', similarity.shape)
print('Done! You can now run app.py')

movies_new.pkl      saved: (10862, 15)
similarity_new.pkl  saved: (10862, 10862)
Done! You can now run app.py
